# 🧠 The Corporate Brain: Week 7 — The Clinical Contender
## Phase 1: Full-Scale Continuous Pre-training (CPT) of the Parametric Layer

### 🏗️ Architectural Objective
This week, we implement the first half of our **Week 8 Bake-Off**. We are building the **"Clinical Contender"**, a commodity LLM that has undergone a full cycle of **Continuous Pre-training (CPT)** on specialized neurological data. 

While the "Missing Middle" is often solved by Knowledge Graphs, this experiment tests the alternative: Can we bake specialized domain knowledge directly into the model's weights to eliminate **Contextual Blindness** at the source?

---

### 🔬 The Contender Blueprint
In this "Glass Box" session, we will:
1. **Substrate Engineering:** Mix **85% Clinical Neurology Data** with **15% General Replay Data** to prevent architectural collapse.
2. **Tokenizer Expansion Analysis:** Audit how the base model perceives clinical jargon and decide on vocabulary extension.
3. **Full CPT Implementation:** Use **QLoRA** to perform a complete training run, effectively "teaching" the model a new technical dialect.
4. **Baseline Evaluation:** Ask the model the same "Semantic Bridge" questions we have prepared for the Week 8 Bake-off to establish its standalone performance.

---

### 📉 Success Metrics (The KPI Dashboard)
* **Training Convergence:** Monitoring the cross-entropy loss to ensure the model is actually learning the new distribution.
* **Knowledge Retention:** Testing general reasoning post-training to ensure zero **Catastrophic Forgetting**.
* **Zero-Shot Acronym Expansion:** Testing the model's new internal "dictionary" (e.g., *TIA, MS, ALS, CVA*).

---

### 🛠️ Step 1: Substrate Preparation (The ETL Phase)
We begin by engineering our training data. We are treating this as a high-fidelity data substrate, ensuring the mix ratio is architecturally sound to balance new knowledge with core linguistic stability.

In [ ]:
# ==============================================================================
# STEP 0: KAGGLE API SETUP
# Target: Automate dataset acquisition for a reproducible pipeline
# ==============================================================================

import os
from dotenv import load_dotenv

# 1. Environment Configuration
#Make sure you have a .env file with your kaggle credentials
load_dotenv()

# 2. Install and Download
%pip install -q kaggle
!kaggle datasets download -d chaitanyakck/medical-text
!unzip -o medical-text.zip

print("✅ Dataset 'train.dat' is now available in your local directory.")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Dataset URL: https://www.kaggle.com/datasets/chaitanyakck/medical-text
License(s): CC0-1.0
100%|██████████████████████████████████████| 11.6M/11.6M [00:00<00:00, 15.5MB/s]

Archive:  medical-text.zip
  inflating: test.dat                
  inflating: train.dat               
✅ Dataset 'train.csv' is now available in your local directory.


In [7]:
# ==============================================================================
# STEP 1: SUBSTRATE PREPARATION (DAT to JSONL)
# Target: Convert tab-separated .dat files into a 85/15 Clinical/General mix
# ==============================================================================

import pandas as pd
import json
import random
import os
from datasets import load_dataset
from tqdm import tqdm

def prepare_cpt_substrate(medical_dat_path, output_file="data/train.jsonl"):
    print("🚀 Initializing Substrate Engineering from .dat files...")
    os.makedirs("data", exist_ok=True)
    
    # 1. Extraction & Schema Mapping
    # Label Mapping based on dataset documentation:
    # 1: Digestive, 2: Cardiovascular, 3: Neoplasms, 4: Nervous System, 5: General
    try:
        # The file is tab-separated, no header
        df = pd.read_csv(medical_dat_path, sep='\t', header=None, names=['label', 'text'])
        
        # Filter for 'Nervous System Diseases' (Label 4)
        neuro_data = df[df['label'] == 4]['text'].tolist()
        print(f"✅ Extracted {len(neuro_data)} Neurology abstracts from .dat substrate.")
    except Exception as e:
        print(f"❌ Error reading .dat file: {e}")
        return

    # 2. Extraction: General Replay Data (The 'Regularizer')
    print("📡 Fetching general replay data (WikiText)...")
    wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    general_pool = [item['text'] for item in wiki if len(item['text']) > 300]
    
    # Mix Logic: 85% Domain, 15% General
    replay_count = int(len(neuro_data) * 0.15)
    general_data = random.sample(general_pool, min(replay_count, len(general_pool)))

    # 3. Transformation: Shuffling to ensure learning stability
    combined_data = neuro_data + general_data
    random.shuffle(combined_data)
    
    # 4. Loading: Writing to JSONL for the MLX Trainer
    with open(output_file, 'w', encoding='utf-8') as f:
        for text in tqdm(combined_data, desc="Building Substrate"):
            clean_text = " ".join(str(text).split())
            if len(clean_text) > 100:
                f.write(json.dumps({"text": clean_text}) + '\n')

    print(f"✨ Substrate Engineering Complete. File saved to: {output_file}")

# --- EXECUTION ---
# Ensure 'train.dat' is in your directory from Step 0
prepare_cpt_substrate("train.dat")

🚀 Initializing Substrate Engineering from .dat files...
✅ Extracted 3051 Neurology abstracts from .dat substrate.
📡 Fetching general replay data (WikiText)...


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Building Substrate: 100%|██████████| 3508/3508 [00:00<00:00, 81924.38it/s]

✨ Substrate Engineering Complete. File saved to: data/train.jsonl


## 📉 Step 2: The Baseline Quiz (Pre-Training Evaluation)
To "walk the walk," we must first witness the base model's **Contextual Blindness**. We are running a "Glass Box" baseline test to record how the un-tuned Llama-3-8B handles dense clinical neurology. 

**The Goal:** Document hallucinations and "General Knowledge" gaps that we aim to fix with Continuous Pre-training in Step 3.

In [8]:
try:
    import mlx_lm
    import mlx.core as mx
    print(f"✅ MLX-LM is ready.")
    print(f"💻 M4 GPU detected. Available Memory: {mx.device_info()['memory_size']/1024**3:.2f} GB")
except ImportError:
    print("❌ MLX-LM not found. Please run `pip install -U mlx-lm`.")

✅ MLX-LM is ready.
💻 M4 GPU detected. Available Memory: 24.00 GB


In [ ]:
# ==============================================================================
# STEP 2: THE BASELINE QUIZ (PRE-TRAINING EVALUATION)
# Target: Document 'Contextual Blindness' in the base Llama-3-8B model
# ==============================================================================

from mlx_lm import load, generate
from huggingface_hub import login
from mlx_lm.sample_utils import make_sampler

# Log in to suppress warnings and enable faster downloads
#login(token=os.environ["HF_TOKEN"])
#print("✅ Hugging Face authentication successful. Speed limits lifted.")

# 1. Load the Base Model
model_path = "mlx-community/Meta-Llama-3-8B-Instruct-4bit"
model, tokenizer = load(model_path)

def run_baseline_quiz(questions):
    print(f"🔬 Querying Base Model: {model_path}\n" + "="*50)

    # Define a deterministic sampler (temp=0.0)
    sampler = make_sampler(temp=0.0)

    results = []
    
    for i, q in enumerate(questions):
        print(f"\n❓ TEST {i+1}: {q[:60]}...")
        
        # FIX: We use 'temp=0' directly in generate for newer mlx-lm versions.
        # If your version is extremely recent, it might prefer sampler_config.
        # This approach is the most stable for M4-optimized builds.
        response = generate(
            model, 
            tokenizer, 
            prompt=q, 
            max_tokens=150, 
            sampler=sampler
        )
        
        print(f"🤖 RESPONSE:\n{response.strip()}")
        results.append(response)
    
    return results

# 2. The 'Clinical Contender' Baseline Questions
# These are specifically designed to trip up a 'General' LLM.
baseline_questions = [
    "Expand the medical acronym 'TIA' in the context of a neurology ward and describe the typical duration of symptoms.",
    "Explain the role of 'Oligodendrocytes' in the central nervous system and name one disease characterized by their destruction.",
    "A patient presents with 'bradykinesia' and 'resting tremor'. What is the most likely neurotransmitter deficiency, and where in the brain is it located?",
    "What is the 'Semantic Gap' between a Slack message saying 'The sensor is drifting' and a Jira ticket marked 'Priority: Blocker' in a medical device firm?"
]

# 3. Execute the Quiz
# ARCHITECT'S NOTE: Save these outputs! We will compare them to the CPT model in Week 8.
baseline_responses = run_baseline_quiz(baseline_questions)

print("\n" + "="*50 + "\n✅ Baseline Quiz Complete. Note any hallucinations or vague 'I am an AI' hedging.")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

🔬 Querying Base Model: mlx-community/Meta-Llama-3-8B-Instruct-4bit

❓ TEST 1: Expand the medical acronym 'TIA' in the context of a neurolo...


TypeError: generate_step() got an unexpected keyword argument 'temp'